In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=SAJJAD;"
    "DATABASE=ACCOUNT_TPS;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

import pandas as pd
query = """
select sum(Debit*Rate) TotalDebit,sum(Credit*Rate) TotalCredit 
from JournalEntries(nolock)
where IsDelete=0 
"""
df = pd.read_sql(query, conn)
df

In [ ]:
from IPython.display import display, HTML
from helpers import show_balance_result

total_debit = df.loc[0, "TotalDebit"]
total_credit = df.loc[0, "TotalCredit"]

if abs(total_debit - total_credit) > 0.01:
    show_balance_result(df, "reports")

In [ ]:
branchQuery = """
select BranchId,sum(Debit*Rate) TotalDebit,sum(Credit*Rate) TotalCredit 
from JournalEntries(nolock)
where IsDelete=0 
Group by BranchId
"""
dfBranch = pd.read_sql(branchQuery, conn)
dfBranch

In [ ]:
tolerance = 0.01

unbalanced_branches = dfBranch.loc[
    (dfBranch["TotalDebit"] - dfBranch["TotalCredit"]).abs() > tolerance,
    ["BranchId", "TotalDebit", "TotalCredit"]
].copy()

if unbalanced_branches.empty:
    show_balance_result(unbalanced_branches, "all branches")
else:
    unbalanced_branches["Difference"] = unbalanced_branches["TotalDebit"] - unbalanced_branches["TotalCredit"]
    show_balance_result(unbalanced_branches, "unbalanced branches")

In [ ]:
branchCurrencyQuery = """
select BranchId,CurrencyId,sum(Debit*Rate) TotalDebit,sum(Credit*Rate) TotalCredit 
from JournalEntries(nolock)
where IsDelete=0 
Group by BranchId,CurrencyId
"""
dfBranchCurrency = pd.read_sql(branchCurrencyQuery, conn)
dfBranchCurrency

In [ ]:
unbalanced_branch_currency = dfBranchCurrency.loc[
    (dfBranchCurrency["TotalDebit"] - dfBranchCurrency["TotalCredit"]).abs() > tolerance,
    ["BranchId", "CurrencyId", "TotalDebit", "TotalCredit"]
].copy()

if unbalanced_branch_currency.empty:
    show_balance_result(unbalanced_branch_currency, "all branch-currency records")
else:
    unbalanced_branch_currency["Difference"] = (
        unbalanced_branch_currency["TotalDebit"] - unbalanced_branch_currency["TotalCredit"]
    )
    show_balance_result(unbalanced_branch_currency, "unbalanced branch-currency records")

In [ ]:
branchCurrencyQuery = """
select BranchId,CurrencyId,YearId,sum(Debit*Rate) TotalDebit,sum(Credit*Rate) TotalCredit 
from JournalEntries(nolock)
where IsDelete=0 
Group by BranchId,CurrencyId,YearId
"""
dfBranchCurrencyYear = pd.read_sql(branchCurrencyQuery, conn)
dfBranchCurrencyYear

In [ ]:
unbalanced_branch_currency_year = dfBranchCurrencyYear.loc[
    (dfBranchCurrencyYear["TotalDebit"] - dfBranchCurrencyYear["TotalCredit"]).abs() > tolerance,
    ["BranchId", "CurrencyId", "YearId", "TotalDebit", "TotalCredit"]
].copy()

if unbalanced_branch_currency_year.empty:
    show_balance_result(unbalanced_branch_currency_year, "all branch-currency-year records")
else:
    unbalanced_branch_currency_year["Difference"] = (
        unbalanced_branch_currency_year["TotalDebit"] - unbalanced_branch_currency_year["TotalCredit"]
    )
    show_balance_result(unbalanced_branch_currency_year, "unbalanced branch-currency-year records")